# CT-SeqTrack B0/B1 regression diagnosis

This notebook is a compact, reproducible companion to the generated report. The generator script reads the immutable run logs and checkpoints and writes `analysis_summary.json`; this notebook checks the decision-critical claims without changing project code or experiment outputs.

In [1]:
import json
from pathlib import Path

cwd = Path.cwd()
candidates = [
    cwd / 'artifacts/ct_checks/reports/20260826_b0_b1_regression_diagnosis/analysis_summary.json',
    cwd / 'reports/20260826_b0_b1_regression_diagnosis/analysis_summary.json',
]
summary_path = next(path for path in candidates if path.exists())
summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary['verdict'], summary['headline']

({'stable_seqtrack_b0_restored': False,
  'b0_core_inherits_seqtrack': True,
  'b0_training_and_evaluator_equivalent_to_seqtrack': False,
  'regression_origin': 'first backward/Adam update',
  'b1_mechanism_regressed': False,
  'ready_for_full_nuscenes': False},
 {'current_b0_success': 29.86980438232422,
  'b0_regression_points': -20.820568084716797,
  'step1_parity': 'FAIL',
  'seqtrack_baseline_status': 'NOT RESTORED',
  'current_b1_mean_delta_cv': -0.2083957368469136})

In [2]:
chain = summary['b0_chain']
assert chain[0]['equal'] and chain[1]['equal'] and chain[2]['equal']
assert not chain[3]['equal']
assert summary['verdict']['regression_origin'] == 'first backward/Adam update'
chain

[{'stage': 'Initialization',
  'prior_high': '798a8def3e82',
  'current_low': '798a8def3e82',
  'equal': True,
  'meaning': 'B0 architecture and seeded initialization match'},
 {'stage': 'First 100 observation fingerprints',
  'prior_high': '100',
  'current_low': '100',
  'equal': True,
  'meaning': 'Candidate IDs, point samples and observation batches match'},
 {'stage': 'First logged B0 loss (step0)',
  'prior_high': '18.006416321',
  'current_low': '18.006416321',
  'equal': True,
  'meaning': 'The first forward and loss reduction match'},
 {'stage': 'B0 parameters after optimizer step1',
  'prior_high': '7f547e5b4afd',
  'current_low': 'cf88c167bc9f',
  'equal': False,
  'meaning': 'The regression begins in backward/Adam, before validation'},
 {'stage': 'First visible loss difference (step3)',
  'prior_high': '15.279096603',
  'current_low': '15.277811050',
  'equal': False,
  'meaning': 'A tiny update difference becomes visible three batches later'},
 {'stage': 'mini_val Success 

In [3]:
mechanism = summary['b1_mechanism']
assert mechanism[0]['learned_minus_cv'] > 0
assert mechanism[-1]['learned_minus_cv'] < 0
assert mechanism[-1]['coverage95'] > mechanism[0]['coverage95']
mechanism

[{'run': 'v25 B1 prior-high@30',
  'epoch': 30,
  'tracking_success': 49.274620056152344,
  'tracking_precision': 52.096282958984375,
  'learned_rmse': 5.6998201458421764,
  'cv_rmse': 5.592396727771393,
  'learned_minus_cv': 0.10742341807078315,
  'help_rate': 0.04208311415044713,
  'nll': 148.56818218750294,
  'coverage95': 0.3114150447133088,
  'deployment_output': 'B0 observation'},
 {'run': 'v25 B1-GRU current',
  'epoch': 30,
  'tracking_success': 30.414661407470703,
  'tracking_precision': 29.844642639160156,
  'learned_rmse': 10.24360913031267,
  'cv_rmse': 10.46341118817519,
  'learned_minus_cv': -0.21980205786251972,
  'help_rate': 0.6109925293489862,
  'nll': 10.170182497165367,
  'coverage95': 0.7796157950907151,
  'deployment_output': 'B0 observation'},
 {'run': 'v25 B1-GRU current',
  'epoch': 60,
  'tracking_success': 31.226476669311523,
  'tracking_precision': 32.286651611328125,
  'learned_rmse': 9.525545757485297,
  'cv_rmse': 9.73394149433221,
  'learned_minus_cv': -